# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Croissant dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells, using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**DOI/Identifier**: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we list all top-level record sets, then fields and columns associated with each record set. All are referenced by their `@id` values as per the Croissant specification.

In [ ]:
# List all available record sets with their @id and name
print("Available RecordSets:")
record_sets = [r for r in metadata.record_sets]
for r in record_sets:
    rec_id = r['@id'] if isinstance(r, dict) else getattr(r, '@id', None)
    rec_name = r.get('name', '') if isinstance(r, dict) else getattr(r, 'name', '')
    print(f"  - @id: {rec_id}, name: {rec_name}")
    # List fields and columns by their @id for this recordset
    if hasattr(r, 'fields') and r.fields:
        print("    Fields:")
        for f in r.fields:
            fid = f['@id'] if isinstance(f, dict) else getattr(f, '@id', None)
            fname = f.get('name', '') if isinstance(f, dict) else getattr(f, 'name', '')
            print(f"      - @id: {fid}, name: {fname}")
    if hasattr(r, 'columns') and r.columns:
        print("    Columns:")
        for c in r.columns:
            cid = c['@id'] if isinstance(c, dict) else getattr(c, '@id', None)
            cname = c.get('name', '') if isinstance(c, dict) else getattr(c, 'name', '')
            print(f"      - @id: {cid}, name: {cname}")

### Preview the First RecordSet

Let's print a few records from the first record set using its `@id`. This will help confirm loading works and illustrate the record structure.

In [ ]:
# Get the first available record set's @id for preview
rs0_id = record_sets[0]['@id'] if isinstance(record_sets[0], dict) else getattr(record_sets[0], '@id', None)

print(f"Preview of first records from record set: {rs0_id}")
cnt = 0
for rec in dataset.records(record_set=rs0_id):
    print(rec)
    cnt += 1
    if cnt >= 3:
        break

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Gather all record set @id values from metadata
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)
    if rs_id: record_set_ids.append(rs_id)

dataframes = {}

for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    if recs:
        dataframes[record_set_id] = pd.DataFrame(recs)
        print(f"Loaded DataFrame for: {record_set_id}, shape: {dataframes[record_set_id].shape}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head(3))

# For downstream sections, pick the first non-empty DataFrame as main
main_rs_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_rs_id = k
        break
if main_rs_id:
    print(f"\nProceeding with main record set: {main_rs_id}\n")

## 4. Exploratory Data Analysis (EDA)
We demonstrate typical data cleaning and transformation steps using fields identified by their `@id` as per FAIR practice.

Let's select a numeric field, filter, normalize, and group as examples.

In [ ]:
# Inspect numeric columns in the primary DataFrame
main_df = dataframes[main_rs_id]
numeric_cols = main_df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
print(f"Numeric columns in main record set ({main_rs_id}): {numeric_cols}")
if not numeric_cols:
    # If no numeric columns detected, try a few common names
    numeric_cols = [col for col in main_df.columns if 'count' in col.lower() or 'coverage' in col.lower() or 'abundance' in col.lower() or 'weight' in col.lower() or 'MW' in col]

# Select the first numeric field for demonstration
if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Using numeric field '{numeric_field}' for filtering and normalization.")
else:
    print("No numeric fields available for EDA.")

if numeric_cols:
    threshold = main_df[numeric_field].mean() # Just for example: use mean as threshold
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: (top 5)")
    display(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records: (first 5)")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a likely categorical column if present
    # Look for a group candidate among object/string columns
    possible_groupby = [col for col in main_df.columns if main_df[col].dtype == object and col.lower() in ['description', 'group', 'category', 'accession', 'protein']]
    group_field = possible_groupby[0] if possible_groupby else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by '{group_field}': (first 5)")
        display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the main numeric field in the main record set, and show a boxplot after normalization.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    sns.set(style="whitegrid")
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

    # Distribution before normalization
    sns.histplot(main_df[numeric_field], bins=30, kde=True, ax=axs[0], color='skyblue')
    axs[0].set_title(f"Distribution of '{numeric_field}' (raw)")

    # After normalization, if available
    if f"{numeric_field}_normalized" in filtered_df.columns:
        sns.boxplot(y=filtered_df[f"{numeric_field}_normalized"], ax=axs[1], color='orange')
        axs[1].set_title(f"Boxplot of '{numeric_field}' (normalized)")
    else:
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        sns.boxplot(y=filtered_df[f"{numeric_field}_normalized"], ax=axs[1], color='orange')
        axs[1].set_title(f"Boxplot of '{numeric_field}' (normalized)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, you've loaded, explored, filtered, and visualized data from the FAIR² Croissant dataset using the `mlcroissant` library. By referencing all data entities using their unique `@id` fields, we've demonstrated reproducible and standards-compliant FAIR data analysis. 

This workflow is a starting point—explore further using specific field `@id`s and additional `mlcroissant` features for deeper analyses relevant to your biomarker or proteomics tasks!